<!-- 학습 보강 셀 -->

# 12. Loading Saved ChromaDB 학습 흐름

이 노트북은 11번에서 저장한 ChromaDB 컬렉션을 다시 불러와 질의하는 예제입니다.
핵심은 원본 문서를 다시 로드하거나 다시 임베딩하지 않고, 저장된 벡터 DB를 재사용한다는 점입니다.

In [1]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [2]:
# OLLAMA_MODEL_PREP_CELL
# Ollama 모델은 pip/requirements.txt로 설치되지 않습니다.
# 이 셀은 노트북 실행 전에 필요한 로컬 Ollama 모델이 있는지 확인하고, 없으면 자동으로 pull 합니다.
import subprocess

OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_LLM_MODEL = 'gemma2:2b'
OLLAMA_EMBED_MODEL = 'nomic-embed-text'

def _installed_ollama_models() -> set[str]:
    try:
        result = subprocess.run(
            ['ollama', 'list'],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError('Ollama CLI가 설치되어 있지 않습니다. https://ollama.com 에서 설치하세요.') from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Ollama 서버가 실행 중인지 확인하세요. 터미널에서 `ollama serve`를 실행하세요.') from exc

    names = set()
    for line in result.stdout.splitlines()[1:]:
        parts = line.split()
        if parts:
            names.add(parts[0])
    return names

def ensure_ollama_model(model_name: str) -> None:
    installed = _installed_ollama_models()
    candidates = {model_name}
    if ':' not in model_name:
        candidates.add(f'{model_name}:latest')

    if installed.intersection(candidates):
        print(f'이미 설치됨: {model_name}')
        return

    print(f'Ollama 모델 다운로드 중: {model_name}')
    subprocess.run(['ollama', 'pull', model_name], check=True)

for model_name in [OLLAMA_LLM_MODEL, OLLAMA_EMBED_MODEL]:
    ensure_ollama_model(model_name)

이미 설치됨: gemma2:2b
이미 설치됨: nomic-embed-text


In [3]:
# LLM과 임베딩 모델 설정
# - 저장할 때 사용한 임베딩 모델과 같은 모델을 사용해야 검색 품질이 유지됩니다.
llm = Ollama(
    model=OLLAMA_LLM_MODEL,
    temperature=0.5,
    request_timeout=120,
    base_url=OLLAMA_BASE_URL,
)

embed_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

In [4]:
# 저장된 ChromaDB 열기
# - 11번 노트북에서 만든 ./chroma_db 디렉토리가 있어야 합니다.
db = chromadb.PersistentClient(
    path='./chroma_db',
)

# 기존 컬렉션 로드
# - get_or_create_collection을 사용하면 컬렉션이 없을 때 빈 컬렉션이 만들어집니다.
# - 실습 편의상 사용하지만, 실제 운영에서는 get_collection으로 존재 여부를 엄격히 확인하는 편이 안전합니다.
chroma_collection = db.get_or_create_collection('quickstart_ollama')

<!-- 학습 보강 셀 -->

## 이 노트북을 실행하기 전 조건

`./chroma_db` 폴더와 `quickstart_ollama` 컬렉션은 11번 노트북을 실행해야 만들어집니다.
폴더가 없거나 컬렉션이 비어 있으면 인덱스 객체는 만들어져도 검색 결과가 없을 수 있습니다.

In [5]:
# ChromaDB 컬렉션을 LlamaIndex vector_store로 감쌉니다.
vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection,
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [6]:
# 저장된 벡터로부터 인덱스 로드
# - from_vector_store는 기존 Chroma 컬렉션을 검색 가능한 VectorStoreIndex로 감쌉니다.
# - 이 단계는 원본 PDF를 다시 읽거나 다시 임베딩하지 않습니다.
index = VectorStoreIndex.from_vector_store(
    vector_store,
    storage_context=storage_context,
    embed_model=embed_model,
)

<!-- 학습 보강 셀 -->

## from_vector_store의 의미

`VectorStoreIndex.from_vector_store()`는 이미 존재하는 벡터 저장소를 LlamaIndex 쿼리 엔진에서 사용할 수 있게 감싸는 단계입니다.
새 임베딩을 만드는 단계가 아니라, 저장된 벡터를 검색 가능한 인터페이스로 연결하는 단계입니다.

In [7]:
# 쿼리 엔진 생성
query_engine = index.as_query_engine(llm=llm)

In [8]:
# 쿼리 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)


질문: 이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘
답변: 이 논문에서 제안하는 모델은 다른 모델과 비교했을 때,  Training cost (FLOPs)를 줄이는 것을 중시하고, 높은 BLEU score를 보여준다는 장점을 가지고 있습니다. 특히, Transformer(big) 모델은 WMT 2014 English-to-German 및 English-to-French 번역 작업에서 다른 모델보다 월등한 성능을 보였습니다. 또한,  Transformer의 base model은 다른 모델들에 비해 빠르게 학습되고, 높은 BLEU score를 유지하는 것을 보여줍니다. 



<!-- 학습 보강 셀 -->

## 로드 성공 여부를 판단하는 기준

답변이 생성되는 것만으로는 충분하지 않습니다.
질문과 관련된 문서가 실제 ChromaDB에서 검색되었는지 확인하려면 필요할 때 `response.source_nodes`를 출력해 파일명과 page_label을 함께 확인하세요.